# Force free flux rope fitting

In [ ]:
import os
import re
import glob
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt

from matplotlib.ticker import AutoMinorLocator
from scipy.special import jv
from lmfit import Parameters, minimize, report_fit

from JunoPy.mag import MagData, mva
from JunoPy.library import find_ticks_between
from JunoPy.spice import InitSpiceKernels, get_juno_position

In [ ]:
start = dt.datetime.strptime('2017-236T20:21:15', '%Y-%jT%H:%M:%S')
end   = dt.datetime.strptime('2017-236T20:21:38', '%Y-%jT%H:%M:%S')

magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
print('Magnetometer file to plot - {:s}'.format(magfile))

# '2017-236T20:21:15 2017-236T20:21:38 NP',
# '2017-338T01:49:57 2017-338T01:51:00 PN'
# '2019-089T20:16:40 2019-089T20:17:00 NP',

In [ ]:
# Perform minimum variance analysis on subset of dataset.
magdata_mva = MagData(filename=magfile)
magdata_mva.df = magdata_mva.df[start:end]
magdata_mva = mva(magdata_mva)

In [ ]:
# Unit normalize the MVA data.
magdata_mva.df['Bmag_saved'] = magdata_mva.df['Bmag']
magdata_mva.df['Bx'] /= magdata_mva.df['Bmag']
magdata_mva.df['By'] /= magdata_mva.df['Bmag']
magdata_mva.df['Bz'] /= magdata_mva.df['Bmag']

## Force free fitting

### 1. Define a flux rope model based on parameters - alpha, B0, ImpactFactor (IF), ThetaAxis, PhiAxis and trajectory of the satellite (eTraj)

In [ ]:
# 1. Define the flux rope model
def model_flux_rope(params, t, eTraj):
    """Creates a flux rope based on parameters."""
    alpha = params['alpha']
    IF = params['IF']
    thetaA = params['thetaA']
    phiA = params['phiA']
    B0 = params['B0']
    
    x = 2 * t * np.sqrt(1 - IF**2)
    r = np.sqrt(IF**2 + x**2)
    
    eZ = np.zeros(3)
    eZ[0] = np.sin(thetaA) * np.cos(phiA)
    eZ[1] = np.sin(thetaA) * np.sin(phiA)
    eZ[2] = np.cos(thetaA)
    
    eY = np.cross(eZ, eTraj)
    eX = np.cross(eY, eZ)
        
    eTrajTangent = eTraj - np.dot(eTraj, eZ) * eZ
    
    B = np.zeros((len(x), 3))
    for i, xi in enumerate(x):
        gamma = np.arctan2(xi, IF)
        eTangent = eTrajTangent * np.cos(gamma) + \
                   np.cross(eZ, eTrajTangent) * np.sin(gamma) + \
                   eZ * np.dot(eZ, eTrajTangent) * (1 - np.cos(gamma))
                   # Rodrigues' Formula

        B_a = B0 * jv(0, alpha * r[i])
        B_t = B0 * jv(1, alpha * r[i])
    
        B[i, :] = B_a * eZ + B_t * eTangent

    return B[:, 0], B[:, 1], B[:, 2]

### 2. From the given MVA dataset, initialize the parameters to be fed into the minimize function.

In [ ]:
def init_params(mvadata):
    """
    """
    thetaA = np.arccos(mvadata.x2[2])
    phiA = np.arctan2(mvadata.x2[1], mvadata.x2[0])
    
    eTraj = np.zeros(3)
    eTraj[0] = mvadata.df['X'][-1] - mvadata.df['X'][0]
    eTraj[1] = mvadata.df['Y'][-1] - mvadata.df['Y'][0]
    eTraj[2] = mvadata.df['Z'][-1] - mvadata.df['Z'][0]

    start = mvadata.df.index[0]
    end = mvadata.df.index[-1]
    mid = start + (end - start) / 2
    T0 = (end - start).total_seconds()
    
    Ti = list()
    for time, row in mvadata.df.iterrows():
        Ti.append((time - mid).total_seconds() / T0)
        
    Ti = np.asarray(Ti)
    
    initParams = Parameters()
    initParams.add('alpha', value=2.4048, min=-2.4048, max=2.4048)
    initParams.add('B0', value=1.0, vary=False)
    initParams.add('IF', value=0.5, min=0., max=1.)
    initParams.add('thetaA', value=thetaA, min=0., max=np.pi)
    initParams.add('phiA', value=phiA, min=-np.pi, max=np.pi)
    
    return Ti, initParams, T0, mid


### 3. Create the objective function

In [ ]:
def objective(params, t, magdata, eTraj):
    """Objective function to minimize."""
    Bx, By, Bz = model_flux_rope(params, t, eTraj)
    
    residuals = np.zeros((3, len(Bx)))
    residuals[0, :] = Bx - magdata.df['Bx'].values
    residuals[1, :] = By - magdata.df['By'].values
    residuals[2, :] = Bz - magdata.df['Bz'].values
    
    # return np.abs(residuals).flatten()
    return np.sum(np.sum(residuals**2, axis=0)) / len(Bx)
    
#     return (Bx - magdata.df['Bx'].values)**2 + \
#            (By - magdata.df['By'].values)**2 + \
#            (Bz - magdata.df['Bz'].values)**2

### 4. Do the fitting.

In [ ]:
Ti, initParams, T0, mid = init_params(magdata_mva)

eTraj = np.zeros(3)
eTraj[0] = magdata_mva.df['X'][-1] - magdata_mva.df['X'][0]
eTraj[1] = magdata_mva.df['Y'][-1] - magdata_mva.df['Y'][0]
eTraj[2] = magdata_mva.df['Z'][-1] - magdata_mva.df['Z'][0]
eTraj = eTraj / np.linalg.norm(eTraj)

out = minimize(objective, initParams, args=(Ti, magdata_mva, eTraj), method='ampgo')

In [ ]:
out

In [ ]:
fitParams = Parameters()
fitParams.add('alpha', value=out.params['alpha'].value, vary=False)
fitParams.add('B0', value=1.0, vary=True)
fitParams.add('IF', value=out.params['IF'].value, vary=False)
fitParams.add('thetaA', value=out.params['thetaA'].value, vary=False)
fitParams.add('phiA', value=out.params['phiA'].value, vary=False)
fitParams

magdata_mva.df['Bx'] *= magdata_mva.df['Bmag']
magdata_mva.df['By'] *= magdata_mva.df['Bmag']
magdata_mva.df['Bz'] *= magdata_mva.df['Bmag']

out2 = minimize(objective, fitParams, args=(Ti, magdata_mva, eTraj), method='cg')

In [ ]:
out2

### 5. Plot and compare with data.

In [ ]:
Bx_fit, By_fit, Bz_fit = model_flux_rope(out2.params, Ti, eTraj)
Bfit = np.asarray([Bx_fit, By_fit, Bz_fit, np.sqrt(Bx_fit**2 + By_fit**2 + Bz_fit**2)])

In [ ]:
def plot_magdata(magdata, start, end, Bfit, out):
    """Plot data between start and end."""
    
    # Slice the magnetometer data.
    # magdata.df = magdata.df.loc[start:end]
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(4, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(11, 8.5), 
                            dpi=300)
    
    axs[:] = magdata.plot(vars=['Bx', 'By', 'Bz', 'Bmag'], axs=axs, Bfit=Bfit)
    
    for a in axs:
        a.set_xlim([timearray[0], timearray[-1]])
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.label_outer()
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    a.legend(['Data', 'Model'], loc='upper left')
    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                 ', DOY={:s}'.format(dt.datetime.strftime(start, '%j') + \
                 ', Reduced $\chi^2$ = {:.2f} nT$^2$'.format(out.residual[0])), y=0.93)
    
    fig.savefig('out_forcefreefit/' + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png')
    # plt.close(fig)

In [ ]:
plot_magdata(magdata_mva, start, end, Bfit, out2)